# Movie Data Processing and Narrative Generation
This notebook loads movie data from multiple streaming platform datasets, removes duplicates, and generates a natural language narrative for each movie.

In [1]:
import pandas as pd
import os

# Load datasets
datasets_dir = 'Datasets'
files = ['amazon_prime_titles.csv', 'disney_plus_titles.csv', 'hulu_titles.csv', 'netflix_titles.csv']

dfs = []
for file in files:
    path = os.path.join(datasets_dir, file)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['source'] = file.split('_')[0]
        dfs.append(df)

# Concatenate all datasets
all_data = pd.concat(dfs, ignore_index=True)
print(f"Total records: {len(all_data)}")

Total records: 22998


In [2]:
# Filter for movies only
movies = all_data[all_data['type'] == 'Movie'].copy()
print(f"Total movies: {len(movies)}")

Total movies: 16481


In [3]:
# Conserve platforms before dropping duplicates
movies['title_clean'] = movies['title'].astype(str).str.strip().str.lower()

# Aggregate the platforms each movie is available on
platforms = movies.groupby('title_clean')['source'].apply(lambda x: ', '.join(x.unique())).reset_index()
platforms.rename(columns={'source': 'available_on'}, inplace=True)

# Drop duplicated movies based on title
movies = movies.drop_duplicates(subset=['title_clean'], keep='first').drop(columns=['source'])

# Merge the aggregated platforms back into the dataframe
movies = movies.merge(platforms, on='title_clean', how='left')
print(f"Total movies after dropping duplicates (with platforms conserved): {len(movies)}")


Total movies after dropping duplicates (with platforms conserved): 15943


In [4]:
# Fill missing values with empty strings for easier string formatting
fill_cols = ['director', 'listed_in', 'description', 'country', 'cast']
movies[fill_cols] = movies[fill_cols].fillna('')

def generate_narrative(row):
    title = row['title']
    genre = row['listed_in'] if row['listed_in'] else "genre-defying"
    director = row['director']
    year = row['release_year']
    desc = row['description']
    
    # Constructing the narrative using f-strings
    if director:
        narrative = f"'{title}' is a {genre} film directed by {director}, released in {year}. The plot follows: {desc}"
    else:
        narrative = f"'{title}' is a {genre} film released in {year}. The plot follows: {desc}"
        
    return narrative

movies['content_narrative'] = movies.apply(generate_narrative, axis=1)

In [5]:
# Show some of the generated narratives
for idx, narrative in enumerate(movies['content_narrative'].head(5)):
    print(f"Narrative {idx + 1}:")
    print(narrative)
    print("-" * 50)

Narrative 1:
'The Grand Seduction' is a Comedy, Drama film directed by Don McKellar, released in 2014. The plot follows: A small fishing village must procure a local doctor to secure a lucrative business contract. When unlikely candidate and big city doctor Paul Lewis lands in their lap for a trial residence, the townsfolk rally together to charm him into staying. As the doctor's time in the village winds to a close, acting mayor Murray French has no choice but to pull out all the stops.
--------------------------------------------------
Narrative 2:
'Take Care Good Night' is a Drama, International film directed by Girish Joshi, released in 2018. The plot follows: A Metro Family decides to fight a Cyber Criminal threatening their stability and pride.
--------------------------------------------------
Narrative 3:
'Secrets of Deception' is a Action, Drama, Suspense film directed by Josh Webber, released in 2017. The plot follows: After a man discovers his wife is cheating on him with a 

In [6]:
from transformers import AutoTokenizer

# Load the tokenizer for the chosen model
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Calculate token lengths for all narratives
# We use add_special_tokens=True because the model will use them during embedding
movies['token_length'] = movies['content_narrative'].apply(lambda x: len(tokenizer.encode(x, add_special_tokens=True)))

# Calculate statistics
max_tokens = movies['token_length'].max()
over_512 = (movies['token_length'] > 512).sum()

print(f"Maximum token length across all narratives: {max_tokens}")
print(f"Number of narratives with more than 512 tokens: {over_512} out of {len(movies)} ({(over_512/len(movies))*100:.2f}%)")

Maximum token length across all narratives: 446
Number of narratives with more than 512 tokens: 0 out of 15943 (0.00%)


In [ ]:
from sentence_transformers import SentenceTransformer
import os

pkl_path = 'movies_with_embeddings.pkl'

if os.path.exists(pkl_path):
    print(f"Loading embeddings from {pkl_path}...")
    movies = pd.read_pickle(pkl_path)
    print("Loaded successfully!")
else:
    print("Generating embeddings... This might take a while.")
    # Load the model
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # Increase the maximum sequence length to 512 tokens
    model.max_seq_length = 512
    print(f"Model max sequence length set to: {model.max_seq_length}")
    
    # Generate embeddings for the narratives
    narratives_list = movies['content_narrative'].tolist()
    embeddings = model.encode(narratives_list, show_progress_bar=True)
    
    # Assign the embeddings back to the dataframe
    movies['embedding'] = list(embeddings)
    
    # Save the dataframe with embeddings to disk
    movies.to_pickle(pkl_path)
    print(f"Embeddings generated and saved to {pkl_path}")


Generating embeddings... This might take a while.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model max sequence length set to: 512


Batches:   0%|          | 0/499 [00:00<?, ?it/s]

Embeddings generated and saved to movies_with_embeddings.pkl
